# Model evaluation and refinement

A single train/test split gives one noisy estimate of performance. This notebook replaces it with
cross-validation, then tames overfitting with **regularisation** and checks the effect with
learning curves. It closes the analysis module with a defensible, reproducible comparison.

## Learning objectives

By the end of this notebook you will be able to:

- estimate performance with K-fold cross-validation and report a spread;
- explain why cross-validation is more trustworthy than one split;
- regularise a linear model with Ridge (L2) and Lasso (L1);
- tune a hyperparameter with a validation loop or `GridSearchCV`;
- read a learning curve to judge whether more data would help.

## Concept

**Cross-validation** splits the training data into K folds, trains on K-1 and validates on the
held-out fold, and repeats K times. The mean score is a more stable estimate than a single split,
and the standard deviation tells you how variable the estimate is. `cross_val_score` does the
loop; a non-negative scorer such as R² is used directly, while error metrics such as RMSE are
negated because scikit-learn maximises scores.

**Regularisation** adds a penalty on coefficient size to the loss, which reduces variance and
guards against overfitting. **Ridge** (L2) shrinks coefficients smoothly toward zero. **Lasso**
(L1) can push some coefficients exactly to zero, performing feature selection. The strength is set
by `alpha`: too small and the model overfits, too large and it underfits. Tune it on the training
folds, never on the test set.

A **learning curve** plots training and validation score against the amount of training data. A
validation curve that is still rising suggests more data would help; a persistent gap suggests the
model is too complex.

## Worked example

### Prepare data and a scaled model

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
import analysis
from ds_practice import load_california, set_seed, regression_metrics, adjusted_r2
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

set_seed(42)
housing = analysis.add_features(load_california())
train, test = train_test_split(housing, test_size=0.2, random_state=42)
X_train, y_train = analysis.split_xy(train)
X_test, y_test = analysis.split_xy(test)
print("train/test rows:", len(X_train), len(X_test))

train/test rows: 16512 4128


### Cross-validation

Five-fold CV on the training set gives a mean and a spread for plain linear regression. RMSE is
negated by the scorer, so we flip the sign back.

In [2]:
pipeline = Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=1e-6))])
rmse_scores = -cross_val_score(pipeline, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error")
r2_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="r2")
print("CV RMSE: ", np.round(rmse_scores, 3), "-> mean", round(rmse_scores.mean(), 3))
print("CV R^2 : ", np.round(r2_scores, 3), "-> mean", round(r2_scores.mean(), 3))

CV RMSE:  [0.669 0.655 0.666 0.663 0.719] -> mean 0.675
CV R^2 :  [0.673 0.669 0.671 0.663 0.621] -> mean 0.659


### Tuning regularisation strength

We sweep `alpha` for Ridge and Lasso and pick the value with the best mean CV RMSE. Lasso also
tells us how many coefficients it zeroed.

In [3]:
rows = []
for name, Model in (("ridge", Ridge), ("lasso", Lasso)):
    for alpha in (0.001, 0.01, 0.1, 1.0, 10.0):
        model = Pipeline([("scale", StandardScaler()), ("model", Model(alpha=alpha, max_iter=5000))])
        scores = -cross_val_score(model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error")
        model.fit(X_train, y_train)
        nonzero = int(np.sum(model.named_steps["model"].coef_ != 0))
        rows.append({"model": name, "alpha": alpha, "cv_rmse": round(scores.mean(), 3), "nonzero_coefs": nonzero})
display(pd.DataFrame(rows).sort_values("cv_rmse").groupby("model").head(2))

,model,alpha,cv_rmse,nonzero_coefs
5,lasso,0.001,0.674,11
4,ridge,10.000,0.674,11
0,ridge,0.001,0.675,11
6,lasso,0.010,0.677,10


### Final fit and honest evaluation

The best variant from the sweep is refit on the full training set and judged once on the test set.
Adjusted R² accounts for the number of predictors.

In [4]:
final = Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=0.1))])
final.fit(X_train, y_train)
test_metrics = regression_metrics(y_test, final.predict(X_test))
print("test metrics:", {k: round(v, 3) for k, v in test_metrics.items()})
print("adjusted R^2:", round(adjusted_r2(test_metrics["r2"], len(y_test), X_test.shape[1]), 3))

test metrics: {'mae': 0.487, 'rmse': 0.674, 'r2': 0.654}
adjusted R^2: 0.653


### Learning curve

Scores as a function of training size show whether the model is data-hungry or capacity-limited.

In [5]:
from sklearn.model_selection import learning_curve

sizes, train_scores, val_scores = learning_curve(
    final, X_train, y_train, cv=3, scoring="r2",
    train_sizes=np.linspace(0.2, 1.0, 5), random_state=42,
)
curve = pd.DataFrame({
    "train_size": sizes,
    "train_r2": train_scores.mean(axis=1).round(3),
    "val_r2": val_scores.mean(axis=1).round(3),
})
display(curve)

,train_size,train_r2,val_r2
0,2201,0.688,-0.127
1,4403,0.679,0.650
2,6604,0.678,0.650
3,8806,0.669,0.654
4,11008,0.666,0.656


## Exercises

1. **Lasso selection.** Fit a Lasso with a tuned alpha and list which features it drives to zero.
   Are they the same features you would have dropped by hand?
2. **Scoring choice.** Re-run the CV with `scoring="neg_mean_absolute_error"` and compare the
   ranking of Ridge and Lasso with the RMSE ranking.
3. **Learning curve reading.** Using the curve above, state whether collecting more block groups
   would plausibly improve the validation score, and justify your answer from the gap between the
   two lines.

## Limitations

Cross-validation assumes the folds are exchangeable; spatial data violates this because nearby
block groups are similar, so a random split leaks geography and reports optimistic scores. A
grouped or spatial split would be more honest. Regularisation improves the bias-variance trade-off
but does not fix a capped target or missing variables. Hyperparameter search over a small grid can
still overfit the validation folds if run enough times, so the final test score is the only number
worth reporting.